# Production Optimization

## Objective
Determine the recommended production quantity for each SKU to minimize unmet demand while respecting the available production capacity.

## Inputs
- Monthly demand forecasts
- Current inventory
- Safety stock
- Monthly production capacity

## Outputs
- Recommended production quantity per SKU
- Remaining unmet demand
- Production capacity utilization
- Optimization summary

## Methodology
A Linear Programming model is used to minimize unmet demand and unnecessary production, subject to production capacity and non-negativity constraints.

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.optimize import linprog

In [2]:
DATA_DIR = Path("streamlit_data")

catalog_df = pd.read_csv(DATA_DIR / "Product_Catalog.csv")
forecast_df = pd.read_csv(DATA_DIR / "Monthly_Forecasts.csv")
inventory_df = pd.read_csv(DATA_DIR / "Inventory_Planning.csv")

print("Product Catalog:", catalog_df.shape)
print("Monthly Forecasts:", forecast_df.shape)
print("Inventory Planning:", inventory_df.shape)

Product Catalog: (59, 2)
Monthly Forecasts: (177, 3)
Inventory Planning: (177, 9)


In [3]:
PLANNING_MONTH = pd.Timestamp("2026-11-01")

MONTHLY_PRODUCTION_CAPACITY = 30_000

SHORTAGE_PENALTY = 1_000
PRODUCTION_COST_WEIGHT = 1

In [4]:
forecast_df["Forecast_Month"] = pd.to_datetime(
    forecast_df["Forecast_Month"]
)

forecast_month_df = forecast_df.loc[
    forecast_df["Forecast_Month"].dt.to_period("M")
    == PLANNING_MONTH.to_period("M")
].copy()

forecast_month_df["Forecast_Demand"] = (
    pd.to_numeric(
        forecast_month_df["Forecast_Demand"],
        errors="coerce"
    )
    .fillna(0)
    .clip(lower=0)
)

forecast_month_df = (
    forecast_month_df
    .groupby("SKU", as_index=False)
    .agg(
        Forecast_Demand=("Forecast_Demand", "sum")
    )
)

if forecast_month_df.empty:
    raise ValueError(
        f"No forecasts found for {PLANNING_MONTH:%Y-%m}"
    )

print("Planning Month:", PLANNING_MONTH.strftime("%B %Y"))
print("Number of SKUs:", len(forecast_month_df))
print(
    "Total Forecast Demand:",
    round(forecast_month_df["Forecast_Demand"].sum(), 2)
)

Planning Month: November 2026
Number of SKUs: 59
Total Forecast Demand: 43427.43


In [5]:
inventory_df["Forecast_Month"] = pd.to_datetime(
    inventory_df["Forecast_Month"]
)

inventory_month_df = inventory_df.loc[
    inventory_df["Forecast_Month"].dt.to_period("M")
    == PLANNING_MONTH.to_period("M")
].copy()

required_columns = [
    "SKU",
    "Current_Stock",
    "Safety_Stock"
]

missing_columns = [
    col for col in required_columns
    if col not in inventory_month_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in Inventory_Planning.csv: {missing_columns}"
    )

inventory_month_df["Current_Stock"] = pd.to_numeric(
    inventory_month_df["Current_Stock"],
    errors="coerce"
)

inventory_month_df["Safety_Stock"] = pd.to_numeric(
    inventory_month_df["Safety_Stock"],
    errors="coerce"
)

inventory_month_df = (
    inventory_month_df
    .groupby("SKU", as_index=False)
    .agg(
        Current_Stock=("Current_Stock", "first"),
        Safety_Stock=("Safety_Stock", "first")
    )
)

if inventory_month_df[
    ["Current_Stock", "Safety_Stock"]
].isna().any().any():
    raise ValueError(
        "Missing inventory or safety-stock values. "
        "Resolve them before optimization."
    )

optimization_df = forecast_month_df.merge(
    inventory_month_df,
    on="SKU",
    how="left",
    validate="one_to_one"
)

if optimization_df["Current_Stock"].isna().any():
    missing_skus = optimization_df.loc[
        optimization_df["Current_Stock"].isna(),
        "SKU"
    ].tolist()

    raise ValueError(
        f"Inventory is missing for these SKUs: {missing_skus}"
    )

optimization_df["Net_Requirement"] = (
    optimization_df["Forecast_Demand"]
    + optimization_df["Safety_Stock"]
    - optimization_df["Current_Stock"]
).clip(lower=0)

display(optimization_df.head())

print(
    "Total Net Requirement:",
    round(optimization_df["Net_Requirement"].sum(), 2)
)

,SKU,Forecast_Demand,Current_Stock,Safety_Stock,Net_Requirement
0,26115-1,211.032883,288,349,272.032883
1,26115-2,329.160004,588,714,455.160004
2,26115-3,308.465424,719,88,0.000000
3,26115-4,1147.427246,1061,657,743.427246
4,26115-5,377.440887,583,169,0.000000


Total Net Requirement: 22485.78


In [6]:
n = len(optimization_df)

net_requirement = optimization_df[
    "Net_Requirement"
].to_numpy(dtype=float)

# Decision variables:
# [Production_1, ..., Production_n,
#  Shortage_1, ..., Shortage_n]

objective = np.concatenate([
    np.full(n, PRODUCTION_COST_WEIGHT),
    np.full(n, SHORTAGE_PENALTY)
])

# Production_i + Shortage_i >= Net_Requirement_i
# Converted to:
# -Production_i - Shortage_i <= -Net_Requirement_i

A_ub = []
b_ub = []

for i in range(n):
    row = np.zeros(2 * n)

    row[i] = -1
    row[n + i] = -1

    A_ub.append(row)
    b_ub.append(-net_requirement[i])

# Total production <= monthly capacity

capacity_row = np.zeros(2 * n)
capacity_row[:n] = 1

A_ub.append(capacity_row)
b_ub.append(MONTHLY_PRODUCTION_CAPACITY)

# Production_i <= Net_Requirement_i
# Prevent unnecessary overproduction in this model.

for i in range(n):
    row = np.zeros(2 * n)
    row[i] = 1

    A_ub.append(row)
    b_ub.append(net_requirement[i])

A_ub = np.asarray(A_ub)
b_ub = np.asarray(b_ub)

bounds = (
    [(0, None)] * n
    + [(0, None)] * n
)

result = linprog(
    c=objective,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=bounds,
    method="highs"
)

if not result.success:
    raise RuntimeError(
        f"Optimization failed: {result.message}"
    )

production = result.x[:n]
shortage = result.x[n:]

print("Optimization Status:", result.message)
print("Objective Value:", round(result.fun, 2))

Optimization Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Objective Value: 22485.78


In [7]:
optimization_results = optimization_df.copy()

optimization_results["Recommended_Production"] = production
optimization_results["Remaining_Shortage"] = shortage

optimization_results["Available_After_Production"] = (
    optimization_results["Current_Stock"]
    + optimization_results["Recommended_Production"]
)

optimization_results["Unmet_Demand_After_Optimization"] = (
    optimization_results["Remaining_Shortage"]
)

optimization_results["Production_Utilization_Pct"] = (
    optimization_results["Recommended_Production"]
    / MONTHLY_PRODUCTION_CAPACITY
    * 100
)

optimization_results["Production_Status"] = np.select(
    [
        optimization_results["Recommended_Production"] > 0,
        optimization_results["Net_Requirement"] == 0
    ],
    [
        "Production Required",
        "No Production Required"
    ],
    default="Capacity Not Allocated"
)

optimization_results = optimization_results.merge(
    catalog_df[["SKU", "Product_Name"]].drop_duplicates("SKU"),
    on="SKU",
    how="left",
    validate="one_to_one"
)

optimization_results = optimization_results[
    [
        "SKU",
        "Product_Name",
        "Forecast_Demand",
        "Current_Stock",
        "Safety_Stock",
        "Net_Requirement",
        "Recommended_Production",
        "Remaining_Shortage",
        "Production_Status"
    ]
].sort_values(
    "Remaining_Shortage",
    ascending=False
).reset_index(drop=True)

display(optimization_results.head(20))

,SKU,Product_Name,Forecast_Demand,Current_Stock,Safety_Stock,Net_Requirement,Recommended_Production,Remaining_Shortage,Production_Status
0,26115-1,Summer children's boys-Sonic,211.032883,288,349,272.032883,272.032883,-0.0,Production Required
1,26115-2,Summer children's boys-Black and orange,329.160004,588,714,455.160004,455.160004,-0.0,Production Required
2,26115-3,Summer children's boys-Black and turquoise,308.465424,719,88,0.000000,0.000000,0.0,No Production Required
3,26115-4,Summer children's boys-SpongeBob,1147.427246,1061,657,743.427246,743.427246,-0.0,Production Required
4,26115-5,Summer children's girls-Hello kitty Rose & tur...,377.440887,583,169,0.000000,0.000000,0.0,No Production Required
5,26115-6,Summer children's girls-turtle,313.373352,5,416,724.373352,724.373352,-0.0,Production Required
6,26115-7,Summer children's girls-Hello kitty black&white,428.786743,223,261,466.786743,466.786743,-0.0,Production Required
7,26115-8,Summer children's girls-Happy,1073.047119,597,545,1021.047119,1021.047119,-0.0,Production Required
8,CAT-013,Boys' summer t-shirt-Beige,219.638596,642,104,0.000000,0.000000,0.0,No Production Required
9,CAT-014,Boys' summer t-shirt-Black,361.648682,596,708,473.648682,473.648682,-0.0,Production Required


In [8]:
total_forecast = optimization_results[
    "Forecast_Demand"
].sum()

total_net_requirement = optimization_results[
    "Net_Requirement"
].sum()

total_production = optimization_results[
    "Recommended_Production"
].sum()

total_shortage = optimization_results[
    "Remaining_Shortage"
].sum()

capacity_utilization = (
    total_production / MONTHLY_PRODUCTION_CAPACITY * 100
    if MONTHLY_PRODUCTION_CAPACITY > 0
    else 0
)

summary_df = pd.DataFrame({
    "Metric": [
        "Planning Month",
        "Total Forecast Demand",
        "Total Net Requirement",
        "Recommended Production",
        "Remaining Shortage",
        "Monthly Production Capacity",
        "Capacity Utilization (%)",
        "SKUs Requiring Production",
        "SKUs With Remaining Shortage"
    ],
    "Value": [
        PLANNING_MONTH.strftime("%B %Y"),
        round(total_forecast, 2),
        round(total_net_requirement, 2),
        round(total_production, 2),
        round(total_shortage, 2),
        MONTHLY_PRODUCTION_CAPACITY,
        round(capacity_utilization, 2),
        int(
            (
                optimization_results["Recommended_Production"] > 0
            ).sum()
        ),
        int(
            (
                optimization_results["Remaining_Shortage"] > 0
            ).sum()
        )
    ]
})

display(summary_df)

,Metric,Value
0,Planning Month,November 2026
1,Total Forecast Demand,43427.43
2,Total Net Requirement,22485.78
3,Recommended Production,22485.78
4,Remaining Shortage,0.0
5,Monthly Production Capacity,30000
6,Capacity Utilization (%),74.95
7,SKUs Requiring Production,30
8,SKUs With Remaining Shortage,0


In [9]:
OUTPUT_DIR = Path("streamlit_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

optimization_results.to_csv(
    OUTPUT_DIR / "Production_Optimization.csv",
    index=False
)

summary_df.to_csv(
    OUTPUT_DIR / "Optimization_Summary.csv",
    index=False
)

print("Optimization files saved successfully.")

print(
    "Production_Optimization.csv:",
    optimization_results.shape
)

print(
    "Optimization_Summary.csv:",
    summary_df.shape
)

Optimization files saved successfully.
Production_Optimization.csv: (59, 9)
Optimization_Summary.csv: (9, 2)
